# Codificacion de datos nominales

In [4]:
import numpy as np
from sklearn.preprocessing import LabelBinarizer, MultiLabelBinarizer

feat = np.array([["Texas"], ["California"], ["Texas"], ["Simon"]])

## TRANSFORMACION
one_hot = LabelBinarizer()
one_hot.fit_transform(feat)

array([[0, 0, 1],
       [1, 0, 0],
       [0, 0, 1],
       [0, 1, 0]])

In [5]:
one_hot.classes_

array(['California', 'Simon', 'Texas'], dtype='<U10')

In [6]:
import pandas as pd
dataframe = pd.DataFrame({"Score": ["Low", "Low", "Medium", "Medium", "High"]})

## generacion de un map
scale_mapper = {"Low":1,
"Medium":2,
"High":3}

dataframe["Score"].replace(scale_mapper)

<ipython-input-6-ac4364ed2f05>:7: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  dataframe["Score"].replace(scale_mapper)


,Score
0,1
1,1
2,2
3,2
4,3


# Diccionario


In [8]:
from sklearn.feature_extraction import DictVectorizer

data_dict = [{"Red": 2, "Blue": 4},
{"Red": 4, "Blue": 3},
{"Red": 1, "Yellow": 2},
{"Red": 2, "Yellow": 2}]

diccionario = DictVectorizer(sparse=False)
feat = diccionario.fit_transform(data_dict)
feat

array([[4., 2., 0.],
       [3., 4., 0.],
       [0., 1., 2.],
       [0., 2., 2.]])

In [11]:
nombres =diccionario.get_feature_names_out()
nombres

array(['Blue', 'Red', 'Yellow'], dtype=object)

# Trabajo con texto

In [21]:
text_data = [" Interrobang. By Aishwarya Henriette",
"Parking And Going. By Karl Gautier                 e!11",
             "simon a sd s d ss"]

strip_ws = [string.strip() for string in text_data]
strip_ws

quitar_puntos = [string.replace(".", "") for string in strip_ws]
quitar_puntos

def minuscula(string: str) -> str:
  return string.lower()

minuscula_todo = [minuscula(string) for string in quitar_puntos]
minuscula_todo

import re
def eliminar_caracteres(string: str) -> str:
  return re.sub(r"[^a-zA-Z0-9 \n\.]", "X", string)


quitaTodo = [eliminar_caracteres(string) for string in minuscula_todo]
quitaTodo

['interrobang by aishwarya henriette',
 'parking and going by karl gautier                 eX11',
 'simon a sd s d ss']

#Remover puntuacion

In [23]:
import unicodedata
import sys
text = ['Hi!!!! I. Love. This. Song....',
'10000% Agree!!!! #LoveIT',
'Right?!?!']


puntuacion = dict.fromkeys(
    (i for i in range(sys.maxunicode)

     if (unicodedata.category(chr(i)).startswith("P"))
     ), None


)


valores = [string.translate(puntuacion) for string in text]
valores

['Hi I Love This Song', '10000 Agree LoveIT', 'Right']

In [29]:
!pip install nltk


NameError: name 'nltk' is not defined

In [36]:
import nltk
nltk.download('punkt_tab')
nltk.download('stopwords')
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize


string = "The science of today is the technology of tomorrow"

tokens = word_tokenize(string)
tokens

# tokenizar y luego quitar palabras

stop = stopwords.words("english")

palabras = [palabra for palabra in tokens if palabra not in stop]
palabras

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


['The', 'science', 'today', 'technology', 'tomorrow']

## importancia de palabras
PARA ESO SE USA TD IDF para determinar la cantidad o valor de utilizacion


In [43]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

text_data = np.array(['I love Brazil. Brazil!',
'Sweden is best',
'Germany beats both'])

tdidf = TfidfVectorizer()
valor =tdidf.fit_transform(text_data)

valor.toarray()

array([[0.        , 0.        , 0.        , 0.89442719, 0.        ,
        0.        , 0.4472136 , 0.        ],
       [0.        , 0.57735027, 0.        , 0.        , 0.        ,
        0.57735027, 0.        , 0.57735027],
       [0.57735027, 0.        , 0.57735027, 0.        , 0.57735027,
        0.        , 0.        , 0.        ]])

In [40]:
tdidf.vocabulary_

{'love': 6,
 'brazil': 3,
 'sweden': 7,
 'is': 5,
 'best': 1,
 'germany': 4,
 'beats': 0,
 'both': 2}

## similitud del coseno

In [51]:
from sklearn.metrics.pairwise import linear_kernel

nuevoTEXto = "is Sweden the love?"

vectorNuevo = tdidf.transform([nuevoTEXto])

similitdCoseno = linear_kernel(valor, vectorNuevo).flatten()

relacion = similitdCoseno.argsort()[::-1][:3]
print([(text_data[i], similitdCoseno[i] )for i in relacion])

[(np.str_('Sweden is best'), np.float64(0.6666666666666666)), (np.str_('I love Brazil. Brazil!'), np.float64(0.2581988897471611)), (np.str_('Germany beats both'), np.float64(0.0))]


#Analisis de sentimiento

In [57]:
from transformers import pipeline

clasificacion = pipeline("sentiment-analysis")

oracion1 = clasificacion("Puta que mierda es todo esto que ha pasado")
oracion2 = clasificacion("Los dias han sido hermosos")
oracion3 = clasificacion("today i ate bologese pasta and was very tasty")


print(oracion1, oracion2, oracion3)

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cpu


[{'label': 'NEGATIVE', 'score': 0.9148590564727783}] [{'label': 'POSITIVE', 'score': 0.949017345905304}] [{'label': 'POSITIVE', 'score': 0.9996581077575684}]
